# Radish Bank Iris Workshop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/redis/redis-iris-demos/blob/main/notebooks/radish_bank_workshop.ipynb)

Build a simple Radish Bank chatbot with **Context Retriever**, **Agent Memory**, and **LangCache**.

Run cells **top to bottom** in one session (~2 hours). Works locally (Jupyter / VS Code) or on Colab.

1. Setup — install packages, paste credentials
2. Define Context Surface models and create the surface
3. Seed Context Retriever, Agent Memory, and LangCache
4. Explore each product, then run the chat loop


# Setup

## Install Packages


In [20]:
# Install all workshop dependencies quietly (-q). Re-run after changing version pins.
%pip install -q "context-surfaces>=0.0.1" "httpx>=0.28.0" "langchain-openai>=1.1.12" "openai>=1.68.0" "pydantic>=2.10.0"

import sys  # Confirm which Python kernel is running (useful on Colab vs local)
print(f"Ready ({sys.executable})")



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Ready (/Users/hillary.toh/.pyenv/versions/3.12.5/bin/python)


### Grab workshop files (if Colab)

**Local:** skip the next cell if you already cloned `redis-iris-demos` and opened this notebook from `notebooks/`.

In [21]:
# NBVAL_SKIP  # Tells automated notebook tests to skip this cell
import sys

# Colab has no repo on disk — clone it. Local Jupyter/VS Code already has the checkout.
if "google.colab" in sys.modules:
    !git clone --depth 1 https://github.com/redis/redis-iris-demos.git  # Shallow clone (faster)
    %cd redis-iris-demos  # Move into the repo so later cells find workshop files
else:
    print("Local: skipped clone — use your existing repo checkout.")

Local: skipped clone — use your existing repo checkout.


In [22]:
import sys
from pathlib import Path

def _find_notebooks_dir(start: Path) -> Path:
    """Find the notebooks directory that contains workshop_helpers.py."""
    matches = []
    for candidate in [start, *start.parents]:  # Check cwd, then parent folders
        if (candidate / "workshop_helpers.py").is_file():
            matches.append(candidate)
        elif (candidate / "notebooks" / "workshop_helpers.py").is_file():
            matches.append(candidate / "notebooks")
    return matches[-1] if matches else start

NOTEBOOKS_DIR = _find_notebooks_dir(Path.cwd())
for p in (NOTEBOOKS_DIR, NOTEBOOKS_DIR.parent):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))  # Enable `import workshop_helpers`

NOTEBOOKS_DIR  # Display detected notebooks path


PosixPath('/Users/hillary.toh/Documents/Playground/redis-iris-demos')

## Step 1 — Configure credentials

Paste Redis Cloud + OpenAI values into `WORKSHOP_CONFIG`.


In [ ]:
# Paste your credentials as strings. Empty values will fail validation below.
WORKSHOP_CONFIG = {
    "OPENAI_API_KEY": "",        # OpenAI key — powers chat and embeddings
    "REDIS_HOST": "",            # Redis Cloud hostname
    "REDIS_PORT": "",            # Cloud port
    "REDIS_PASSWORD": "",        # Redis Cloud password
    "REDIS_SSL": "true",         # "true" for Redis Cloud TLS
    "CTX_ADMIN_KEY": "",         # Context Retriever admin API key
    "MEMORY_API_BASE_URL": "",   # Agent Memory REST base URL
    "MEMORY_STORE_ID": "",       # Your memory store ID
    "MEMORY_API_KEY": "",        # Agent Memory API key
    "LANGCACHE_HOST": "",        # LangCache server host
    "LANGCACHE_CACHE_ID": "",    # LangCache cache ID
    "LANGCACHE_API_KEY": "",     # LangCache API key
}

from workshop_helpers import validate_env

env_values = validate_env(WORKSHOP_CONFIG)  # Raises if any required key is missing
print("Credentials OK")


Credentials OK


## Step 2 — Context Surface data model

Four `ContextModel` classes → MCP tools. Demo data in `workshop_data/`.


In [24]:
from __future__ import annotations

from typing import Any

# ContextModel turns Pydantic-like classes into Redis-backed entities + MCP tools
from context_surfaces.context_model import ContextField, ContextModel, ContextRelationship


class Customer(ContextModel):

    __redis_key_template__ = "workshop_customer:{customer_id}"  # Redis key pattern for this entity

    customer_id: str = ContextField(
        description="Unique customer identifier; demo customer is always CUST001.",
        is_key_component=True,  # Part of the primary key
    )
    name: str = ContextField(description="Customer full name", index="text", weight=2.0)  # Full-text search
    segment: str = ContextField(description="Segment e.g. retail", index="tag")  # Exact-match filter
    accounts: Any = ContextRelationship(
        description="Deposit accounts for this customer",
        target="Account",  # Links to the Account model
        source_field="customer_id",  # Join on customer_id
    )


class Account(ContextModel):

    __redis_key_template__ = "workshop_account:{account_id}"

    account_id: str = ContextField(description="Account identifier", is_key_component=True)
    customer_id: str = ContextField(description="Owning customer", index="tag")  # Filter by customer
    account_type: str = ContextField(description="savings or current", index="tag")
    balance_sgd: float = ContextField(
        description="Balance in SGD", index="numeric", sortable=True  # Range queries + sorting
    )
    status: str = ContextField(description="active or inactive", index="tag")
    customer: Any = ContextRelationship(
        description="Account owner",
        target="Customer",
        source_field="customer_id",
    )


class FixedDepositPlan(ContextModel):

    __redis_key_template__ = "workshop_fd_plan:{plan_id}"

    plan_id: str = ContextField(
        description="Primary key: FD6 or FD12 in this demo.",
        is_key_component=True,
    )
    tenure_months: int = ContextField(
        description="Tenure in months", index="numeric", sortable=True
    )
    rate_percent: float = ContextField(
        description="Annual rate percent", index="numeric", sortable=True
    )
    min_deposit_sgd: int = ContextField(
        description="Minimum deposit SGD", index="numeric", sortable=True
    )


class BankDocument(ContextModel):

    __redis_key_template__ = "workshop_document:{document_id}"

    document_id: str = ContextField(description="Stable document id", is_key_component=True)
    title: str = ContextField(description="Document title", index="text", weight=2.0)
    category: str = ContextField(description="fd_faq, branch_guide, etc.", index="tag")
    content: str = ContextField(description="Markdown body", index="text")  # Keyword search
    content_embedding: list[float] = ContextField(
        description="Embedding of content for vector search",
        index="vector",  # Semantic similarity search
        vector_dim=1536,  # OpenAI text-embedding-3-small dimension
        distance_metric="cosine",
    )


# All entity classes — passed to setup and seeding steps later
WORKSHOP_ENTITIES = [Customer, Account, FixedDepositPlan, BankDocument]
[cls.__name__ for cls in WORKSHOP_ENTITIES]  # Display registered model names


['Customer', 'Account', 'FixedDepositPlan', 'BankDocument']

## Step 3 — Create Context Surface

Create the surface, agent key, and service clients.


In [ ]:
import uuid  # Generate a unique session ID for short-term chat memory

from workshop_helpers import (
    DEMO_CUSTOMER_ID,       # Always "CUST001" — the demo customer
    chat_turn,              # Run one user message through the full agent pipeline
    init_services,          # Build Context Retriever, Memory, and LangCache clients
    print_turn_result,      # Pretty-print trace + assistant reply
    run_workshop_setup,     # Register models and create the Context Surface in Redis
)

# Reuses the existing surface/key. If you changed the models above, add update_existing=True.
setup_result = await run_workshop_setup(WORKSHOP_ENTITIES)  # Returns surface_id + agent_key
cs_service, memory_service, langcache_service, settings = init_services()
SESSION_ID = f"workshop-{uuid.uuid4()}"  # New thread for session (short-term) memory

print(f"Surface: {setup_result.surface_id}")  # Context Surface ID in Redis
print(f"Session: {SESSION_ID}")               # This chat thread's session ID
print(f"Customer: {DEMO_CUSTOMER_ID}")        # Demo customer used throughout


Surface: 75ea1c41-980e-4642-a939-4b1c6e30a361
Session: workshop-29c4e0d2-654e-41ac-9d9d-5f01175588fd
Customer: CUST001


## Part 1 — Seed Context Retriever data

Edit records below, then run import. `BankDocument` loads from `workshop_data/bank_documents.jsonl`.


In [26]:
# Demo records to import into Context Retriever — edit these to experiment
CONTEXT_DATA = {
    "Customer": [
        {
            "customer_id": "CUST001",
            "name": "Merv Kwok",
            "segment": "retail",
        },
    ],
    "Account": [
        {
            "account_id": "ACC001",
            "customer_id": "CUST001",
            "account_type": "savings",
            "balance_sgd": 42500.0,
            "status": "active",
        },
        {
            "account_id": "ACC002",
            "customer_id": "CUST001",
            "account_type": "current",
            "balance_sgd": 16200.0,
            "status": "active",
        },
    ],
    "FixedDepositPlan": [
        {"plan_id": "FD6", "tenure_months": 6, "rate_percent": 2.8, "min_deposit_sgd": 1000},
        {"plan_id": "FD12", "tenure_months": 12, "rate_percent": 3.1, "min_deposit_sgd": 1000},
    ],
}

# Preview what will be imported (BankDocument comes from a JSONL file instead)
for entity, rows in CONTEXT_DATA.items():
    print(f"{entity}: {len(rows)} record(s)")
print("BankDocument: loaded from workshop_data/bank_documents.jsonl during import")


Customer: 1 record(s)
Account: 2 record(s)
FixedDepositPlan: 2 record(s)
BankDocument: loaded from workshop_data/bank_documents.jsonl during import


In [27]:
from workshop_helpers import seed_workshop_context_data

# Upsert CONTEXT_DATA into Redis and auto-load BankDocument from JSONL
imported = await seed_workshop_context_data(
    WORKSHOP_ENTITIES,  # Model classes defined in Step 2
    CONTEXT_DATA,       # Inline records above
    settings=settings,  # Redis + API credentials
)

print("Imported:")
for entity, count in imported.items():
    print(f"  {entity}: {count}")  # Number of records written per entity


Imported:
  Customer: 1
  Account: 2
  FixedDepositPlan: 2
  BankDocument: 1


## Part 2 — Context Retriever (explore)

After seeding, list MCP tools and call one directly.

**Note:** `filter_*` and `search_*` tools take a single argument named **`value`** (the field to match is in the tool name, e.g. `filter_account_by_customer_id` → `{"value": "CUST001"}`).


In [28]:
# List every MCP tool auto-generated from your ContextModel classes
tools = await cs_service.list_tools()
print(f"{len(tools)} tools:")

for tool in tools:
    print(f"  - {tool['name']}")  # e.g. filter_account_by_customer_id, get_customer_by_id


16 tools:
  - filter_account_by_account_type
  - filter_account_by_customer_id
  - filter_account_by_status
  - filter_bankdocument_by_category
  - filter_customer_by_segment
  - find_account_by_balance_sgd_range
  - find_fixeddepositplan_by_min_deposit_sgd_range
  - find_fixeddepositplan_by_rate_percent_range
  - find_fixeddepositplan_by_tenure_months_range
  - get_account_by_id
  - get_bankdocument_by_id
  - get_customer_by_id
  - get_fixeddepositplan_by_id
  - search_bankdocument_by_content_embedding_similarity
  - search_bankdocument_by_text
  - search_customer_by_text


In [29]:
# Call a filter tool directly — returns all accounts owned by the demo customer
accounts = await cs_service.call_tool(
    "filter_account_by_customer_id",  # Tool name matches the indexed field
    {"value": DEMO_CUSTOMER_ID},      # filter_* tools take a single "value" argument
)
accounts  # Display the JSON response (count, results, pagination)


{'count': 2,
 'has_more': False,
 'limit': 10,
 'offset': 0,
 'results': [{'account_id': 'ACC001',
   'account_type': 'savings',
   'balance_sgd': 42500,
   'customer_id': 'CUST001',
   'id': 'workshop_account:ACC001',
   'status': 'active'},
  {'account_id': 'ACC002',
   'account_type': 'current',
   'balance_sgd': 16200,
   'customer_id': 'CUST001',
   'id': 'workshop_account:ACC002',
   'status': 'active'}],
 'total_count': 2}

## Part 3 — Agent Memory

Seed long-term facts, then search.


In [30]:
# Long-term memories persist across chat sessions — seed a few customer preferences
MEMORY_SEEDS = [
    {
        "memory_id": "workshop-seed-paperless",  # Stable ID (upsert, not duplicate)
        "text": "Prefers paperless statements and online banking",
        "topics": ["banking", "preferences"],  # Optional tags for filtering
    },
    {
        "memory_id": "workshop-seed-fd-funding",
        "text": "When placing a fixed deposit, use the Savings Account as the funding account",
        "topics": ["fixed_deposit", "preferences"],
    },
]

for seed in MEMORY_SEEDS:
    memory_service.create_long_term_memory(
        text=str(seed["text"]),
        owner_id=DEMO_CUSTOMER_ID,  # Memories are scoped to a customer
        memory_type=seed.get("memory_type", "semantic"),  # Default: semantic recall
        topics=list(seed.get("topics") or []),
        memory_id=str(seed["memory_id"]),
    )

print(f"Seeded {len(MEMORY_SEEDS)} long-term memories for {DEMO_CUSTOMER_ID}")
for seed in MEMORY_SEEDS:
    print(f"  - [{seed['memory_id']}] {seed['text']}")


Seeded 2 long-term memories for CUST001
  - [workshop-seed-paperless] Prefers paperless statements and online banking
  - [workshop-seed-fd-funding] When placing a fixed deposit, use the Savings Account as the funding account


In [31]:
# Semantic search over long-term memories (vector similarity, not exact match)
memories = await memory_service.asearch_long_term_memory(
    text="fixed deposit funding account preferences",  # Natural-language query
    owner_id=DEMO_CUSTOMER_ID,  # Only search this customer's memories
    limit=5,  # Top 5 matches
)
for item in memories:
    print(f"- [{item.get('id', '?')}] {item.get('text', item)}")


- [seed-radish-bank-1] When placing a fixed deposit, use the Savings Account as the funding account
- [workshop-seed-fd-funding] When placing a fixed deposit, use the Savings Account as the funding account


## Part 4 — LangCache


In [32]:
# A canned Q&A pair to seed LangCache — avoids an LLM call for common questions
LANGCACHE_PROMPT = "What are your current fixed deposit interest rates?"

LANGCACHE_RESPONSE = """We currently offer two fixed deposit plans:

- **FD6** (6-month term): **2.8% p.a.** — minimum deposit SGD 1,000
- **FD12** (12-month term): **3.1% p.a.** — minimum deposit SGD 1,000

Interest is calculated daily and paid at maturity. Early withdrawal forfeits all accrued interest.
You can open an FD through your account portal or visit any Radish Bank branch."""


In [ ]:
ok = await langcache_service.store(LANGCACHE_PROMPT, LANGCACHE_RESPONSE)  # Cache prompt → response
print(f"LangCache seed: {'OK' if ok else 'FAILED'}")


LangCache seed: OK


Try exact prompt, paraphrase (HIT), unrelated question (MISS).


In [34]:
# Test LangCache: exact match, paraphrase (should HIT), unrelated question (should MISS)
for question in [
    LANGCACHE_PROMPT,  # Exact seeded prompt — similarity ≈ 1.0
    "Tell me about your FD interest rates",  # Paraphrase — semantic HIT
    "What are your branch opening hours?",  # Unrelated — MISS
]:
    result = await langcache_service.search(question)  # Vector similarity search
    if result:
        print(f"HIT ({result['similarity']:.3f}): {question}")
        if question == LANGCACHE_PROMPT:
            print("  (exact seeded prompt)")
    else:
        print(f"MISS: {question}")


HIT (1.000): What are your current fixed deposit interest rates?
  (exact seeded prompt)
HIT (0.906): Tell me about your FD interest rates
MISS: What are your branch opening hours?


## Part 5 — Chat loop

Each turn runs: **LangCache → session memory → long-term memory → LLM + MCP tools**.

Edit the messages below, or add more `chat_turn` cells.


In [35]:
# Turn 1: FD rates question — expect LangCache HIT (no LLM call)
turn1 = await chat_turn(
    "Tell me about your fixed deposit interest rates",  # User message
    session_id=SESSION_ID,  # Short-term memory thread
    cs_service=cs_service,  # Context Retriever MCP tools
    memory_service=memory_service,  # Long-term + session memory
    langcache_service=langcache_service,  # Checked first on every turn
    settings=settings,
)
print_turn_result(turn1)  # Shows trace (LangCache/MCP/memory) + assistant reply


--- trace ---
LangCache HIT (similarity=0.963)
--- assistant ---
We currently offer two fixed deposit plans:

- **FD6** (6-month term): **2.8% p.a.** — minimum deposit SGD 1,000
- **FD12** (12-month term): **3.1% p.a.** — minimum deposit SGD 1,000

Interest is calculated daily and paid at maturity. Early withdrawal forfeits all accrued interest.
You can open an FD through your account portal or visit any Radish Bank branch.


In [36]:
# Turn 2: Account balance — expect LangCache MISS, then MCP tool call
turn2 = await chat_turn(
    "What is my savings account balance?",
    session_id=SESSION_ID,
    cs_service=cs_service,
    memory_service=memory_service,
    langcache_service=langcache_service,
    settings=settings,
)
print_turn_result(turn2)  # Trace should show filter_account_by_customer_id


--- trace ---
LangCache MISS
Session memory: stored USER event
Session memory: 1 event(s) in thread
Long-term memory: 0 match(es)
MCP tool call: filter_account_by_customer_id
Session memory: stored ASSISTANT event
--- assistant ---
Your savings account balance is SGD 42,500.


In [37]:
# Turn 3: FD placement — uses session context + long-term memory + MCP tools
turn3 = await chat_turn(
    "I'd like to place SGD 5,000 in a 6-month fixed deposit. What would potentially be my account balance if I did this?",
    session_id=SESSION_ID,
    cs_service=cs_service,
    memory_service=memory_service,
    langcache_service=langcache_service,
    settings=settings,
)
print_turn_result(turn3)  # Should recall savings balance from turn 2 + FD funding preference


--- trace ---
LangCache MISS
Session memory: stored USER event
Session memory: 3 event(s) in thread
Long-term memory: 2 match(es)
MCP tool call: get_fixeddepositplan_by_id
MCP tool call: get_fixeddepositplan_by_id
Session memory: stored ASSISTANT event
--- assistant ---
You can choose between two fixed deposit plans for your SGD 5,000:

1. **FD6**: 6-month tenure at an interest rate of 2.8%.
2. **FD12**: 12-month tenure at an interest rate of 3.1% (not applicable for your 6-month request).

If you place SGD 5,000 in the 6-month fixed deposit (FD6), the interest earned would be approximately:

- Interest = Principal × Rate × Time
- Interest = 5,000 × 0.028 × (6/12) = SGD 70

Your potential account balance after placing the fixed deposit would be:

- New balance = Current balance - Deposit + Interest
- New balance = 42,500 - 5,000 + 70 = SGD 37,570

So, your potential account balance would be **SGD 37,570** after placing the fixed deposit.


### Restart session

Clear **short-term memory** (this chat thread) and start fresh. Long-term memories from Part 3 are unchanged.


In [38]:
import uuid
import httpx

# Delete short-term (session) memory for the current thread via the Memory API
conn = memory_service.connection(owner_id=DEMO_CUSTOMER_ID)  # API base URL, store ID, key
auth = conn.api_key
if not auth.lower().startswith(("bearer ", "basic ")):
    auth = f"Bearer {auth}"  # Ensure Authorization header format
url = f"{conn.api_base_url}/v1/stores/{conn.store_id}/session-memory/{SESSION_ID}"
async with httpx.AsyncClient(timeout=30.0) as client:
    resp = await client.delete(url, headers={"Authorization": auth})  # Wipe this chat thread
    if resp.status_code not in (200, 204, 404):  # 404 = already empty
        resp.raise_for_status()

SESSION_ID = f"workshop-{uuid.uuid4()}"  # New session — long-term memories are unchanged
print(f"Session restarted: {SESSION_ID}")


Session restarted: workshop-4c58602d-045a-447b-921d-743a97c6d81e
